In [30]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import operator

In [31]:
load_dotenv()
llm = ChatOpenAI()
parser = StrOutputParser()

In [32]:
class EvaluationSchema(BaseModel):
  feedback: str = Field(description='Detailed feedbackfor the essay')
  score: int = Field(description='score out of 10', ge= 0, le=10)

In [33]:
structured_model = llm.with_structured_output(EvaluationSchema)


/Users/abdulbasit/Desktop/Ai Projects Campus X/langgraph/.venv/lib/python3.10/site-packages/langchain_openai/chat_models/base.py:2210: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


In [34]:
essay = """
Democracy is a system in which the people choose their leaders by voting. Pakistan adopted democracy after independence in 1947, but its journey has not been easy. The country has seen many ups and downs because of political instability and military rule. Still, democracy remains the best form of government for Pakistan. It gives people the right to express their opinions and elect representatives of their choice. In a democratic system, leaders are accountable to the public. The Parliament makes laws for the welfare of the nation. Elections are an important part of democracy because they allow peaceful transfer of power.
Pakistan’s democracy has faced problems such as corruption, weak institutions, and lack of political unity. Sometimes governments fail to complete their terms, which affects national progress. However, democratic values are becoming stronger with time. The role of the media, judiciary, and educated youth is very important in strengthening democracy. Public awareness and participation can improve the political system.
A strong democracy brings stability, justice, and development to the country. For Pakistan to progress, all institutions must work together honestly. Democracy can lead Pakistan toward peace and prosperity if it is supported by fair elections and responsible leadership.
"""

In [35]:
class EssayState(TypedDict):
  essay: str
  language_feedback: str
  analysis_feedback: str
  clarity_feedback: str
  overall_feedback: str
  individual_scores: Annotated[list[int], operator.add]
  avg_score: float

In [53]:
def evaluate_language(state: EssayState):
  prompt = PromptTemplate(template= 'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}', input_variables= ['essay'])
  chain = prompt | structured_model
  result = chain.invoke({'essay': state['essay']})

  return {'language_feedback': result.feedback, 'individual_scores': [result.score]}

In [54]:
def evaluate_analysis(state: EssayState):
  prompt = PromptTemplate(template= 'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {essay}', input_variables= ['essay'])
  chain = prompt | structured_model
  result = chain.invoke({'essay': state['essay']})

  return {'analysis_feedback': result.feedback, 'individual_scores': [result.score]}

In [55]:
def evaluate_clarity(state: EssayState):
  prompt = PromptTemplate(template= 'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {essay}', input_variables= ['essay'])
  chain = prompt | structured_model
  result = chain.invoke({'essay': state['essay']})

  return {'clarity_feedback': result.feedback, 'individual_scores': [result.score]}

In [56]:
def final_evaluation(state: EssayState):
  prompt = PromptTemplate(template= "Based on the following feedbacks create a summarized feedback \n language feedback - {language_feedback} \n depth of analysis feedback - {analysis_feedback} \n clarity of thought feedback - {clarity_feedback}", 
                          input_variables= ['language_feedback', 'analysis_feedback', 'clarity_feedback'])
  chain = prompt | structured_model
  output = chain.invoke({'language_feedback': state['language_feedback'], 'analysis_feedback': state['analysis_feedback'], 'clarity_feedback': state['clarity_feedback']})

  avg_score = sum(state['individual_scores']) / len(state['individual_scores'])
  
  return {'overall_feedback': output.feedback, 'avg_score': avg_score}

In [57]:
graph = StateGraph(EssayState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_clarity', evaluate_clarity)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_clarity')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_clarity', 'final_evaluation')

graph.add_edge('final_evaluation', END)
workflow = graph.compile()

In [58]:
initial_state = {
  'essay': essay
}

final_state = workflow.invoke(initial_state)
print(final_state)

{'essay': '\nDemocracy is a system in which the people choose their leaders by voting. Pakistan adopted democracy after independence in 1947, but its journey has not been easy. The country has seen many ups and downs because of political instability and military rule. Still, democracy remains the best form of government for Pakistan. It gives people the right to express their opinions and elect representatives of their choice. In a democratic system, leaders are accountable to the public. The Parliament makes laws for the welfare of the nation. Elections are an important part of democracy because they allow peaceful transfer of power.\nPakistan’s democracy has faced problems such as corruption, weak institutions, and lack of political unity. Sometimes governments fail to complete their terms, which affects national progress. However, democratic values are becoming stronger with time. The role of the media, judiciary, and educated youth is very important in strengthening democracy. Publ